# Data Pull (LCD + Hourly Precip)

Set `NOAA_TOKEN` in .env before running

In [2]:
import io
import os
import time
from pathlib import Path

import pandas as pd
import requests

ModuleNotFoundError: No module named 'pandas'

In [ ]:
START_DATE = "2025-01-01"
END_DATE = "2026-01-01"
TIMEZONE = "America/New_York"

HARD_CODED_STATIONS = [
    "72503014732",  # LaGuardia
    "74486094789",  # JFK
    "72502014734",  # Newark
]

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

env_path = Path("../.env")
if not env_path.exists():
    raise ValueError("Missing ../.env file")

NOAA_TOKEN = None
for raw_line in env_path.read_text().splitlines():
    line = raw_line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    if key.strip() == "NOAA_TOKEN":
        NOAA_TOKEN = value.strip().strip('"').strip("'")
        break

if not NOAA_TOKEN:
    raise ValueError("NOAA_TOKEN not found in ../.env")

In [ ]:
GLOBAL_HOURLY_BASE = "https://www.ncei.noaa.gov/data/global-hourly/access"

In [ ]:
stations_df = pd.DataFrame({"station_id": HARD_CODED_STATIONS})

stations_path = RAW_DIR / "stations.csv"
stations_df.to_csv(stations_path, index=False)

stations_df #just to check

,station_id
0,72503014732
1,74486094789
2,72502014734


In [ ]:
def parse_first_number(value, scale=1.0):
    if pd.isna(value):
        return None
    piece = str(value).split(",", 1)[0].strip()
    if piece in ["", "+9999", "9999", "-9999"]:
        return None
    try:
        return float(piece) / scale
    except ValueError:
        return None


def fetch_station_year_csv(station_id, year):
    url = f"{GLOBAL_HOURLY_BASE}/{year}/{station_id}.csv"
    response = requests.get(url, timeout=60)
    if response.status_code != 200:
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(response.text), low_memory=False)


def pull_hourly_dataset(dataset_id, station_ids, start_date, end_date):
    rows = []
    start_ts = pd.to_datetime(start_date, utc=True)
    end_ts = pd.to_datetime(end_date, utc=True)

    for station_id in station_ids:
        for year in range(start_ts.year, end_ts.year + 1):
            source_df = fetch_station_year_csv(station_id, year)
            if source_df.empty or "DATE" not in source_df.columns:
                continue

            source_df["DATE"] = pd.to_datetime(source_df["DATE"], utc=True, errors="coerce")
            source_df = source_df[(source_df["DATE"] >= start_ts) & (source_df["DATE"] <= end_ts)].copy()
            if source_df.empty:
                continue

            if dataset_id == "NORMAL_HLY":
                for _, row in source_df.iterrows():
                    tmp = parse_first_number(row.get("TMP"), scale=10.0)
                    dew = parse_first_number(row.get("DEW"), scale=10.0)
                    if tmp is not None:
                        rows.append({"station_id": station_id, "date": row["DATE"], "datatype": "TMP", "value": tmp})
                    if dew is not None:
                        rows.append({"station_id": station_id, "date": row["DATE"], "datatype": "DEW", "value": dew})

            if dataset_id == "PRECIP_HLY":
                for _, row in source_df.iterrows():
                    prcp = parse_first_number(row.get("AA1"), scale=10.0)
                    if prcp is not None:
                        rows.append({"station_id": station_id, "date": row["DATE"], "datatype": "PRCP", "value": prcp})

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(rows)

Took 2:15 and 3:10 minutes respectively on my machine:

In [ ]:
HOURLY_DATASET_ID = "NORMAL_HLY"

station_ids = stations_df["station_id"].tolist()
hourly_raw = pull_hourly_dataset(
    dataset_id=HOURLY_DATASET_ID,
    station_ids=station_ids,
    start_date=START_DATE,
    end_date=END_DATE,
)

hourly_path = RAW_DIR / "hourly_meteorology_raw.csv"
if not hourly_raw.empty:
    hourly_raw.to_csv(hourly_path, index=False)

hourly_raw.head()

,station_id,date,datatype,value
0,72503014732,2025-01-01 00:00:00+00:00,TMP,8.9
1,72503014732,2025-01-01 00:00:00+00:00,DEW,3.3
2,72503014732,2025-01-01 00:51:00+00:00,TMP,7.2
3,72503014732,2025-01-01 00:51:00+00:00,DEW,2.8
4,72503014732,2025-01-01 01:51:00+00:00,TMP,7.2


In [ ]:
precip_raw = pull_hourly_dataset(
    dataset_id="PRECIP_HLY",
    station_ids=station_ids,
    start_date=START_DATE,
    end_date=END_DATE,
)

precip_path = RAW_DIR / "hourly_precip_raw.csv"
if not precip_raw.empty:
    precip_raw.to_csv(precip_path, index=False)

precip_raw.head()

,station_id,date,datatype,value
0,72503014732,2025-01-01 01:51:00+00:00,PRCP,0.1
1,72503014732,2025-01-01 02:31:00+00:00,PRCP,0.1
2,72503014732,2025-01-01 02:51:00+00:00,PRCP,0.1
3,72503014732,2025-01-01 02:54:00+00:00,PRCP,0.1
4,72503014732,2025-01-01 03:00:00+00:00,PRCP,0.3


In [ ]:
def make_wide(df, prefix):
    if df.empty:
        return df

    needed_cols = ["station_id", "date", "datatype", "value"]
    existing_cols = [col for col in needed_cols if col in df.columns]

    work_df = df[existing_cols].copy()
    work_df["date"] = pd.to_datetime(work_df["date"], utc=True, errors="coerce")
    work_df = work_df.dropna(subset=["date"])

    wide_df = work_df.pivot_table(
        index=["station_id", "date"],
        columns="datatype",
        values="value",
        aggfunc="mean",
    ).reset_index()

    wide_df.columns.name = None

    for col in list(wide_df.columns):
        if col not in ["station_id", "date"]:
            wide_df = wide_df.rename(columns={col: f"{prefix}_{col}"})

    return wide_df

In [ ]:
hourly_wide = make_wide(hourly_raw, prefix="met") if not hourly_raw.empty else pd.DataFrame()
precip_wide = make_wide(precip_raw, prefix="prcp") if not precip_raw.empty else pd.DataFrame()

if hourly_wide.empty and precip_wide.empty:
    raise RuntimeError("Both hourly and precip pulls are empty. Check dataset IDs, date window, and stations.")

if hourly_wide.empty:
    merged = precip_wide.copy()
elif precip_wide.empty:
    merged = hourly_wide.copy()
else:
    merged = hourly_wide.merge(precip_wide, on=["station_id", "date"], how="outer")

merged = merged.sort_values(["station_id", "date"]).reset_index(drop=True)
merged["date_local"] = merged["date"].dt.tz_convert(TIMEZONE)
merged["hour_local"] = merged["date_local"].dt.hour
merged["month_local"] = merged["date_local"].dt.month

merged_path = PROCESSED_DIR / "minimum_hourly_merged.csv"
merged.to_csv(merged_path, index=False)

merged.head()

,station_id,date,met_DEW,met_TMP,prcp_PRCP,date_local,hour_local,month_local
0,72502014734,2025-01-01 00:00:00+00:00,5.0,10.0,NaN,2024-12-31 19:00:00-05:00,19,12
1,72502014734,2025-01-01 00:51:00+00:00,4.4,10.0,0.1,2024-12-31 19:51:00-05:00,19,12
2,72502014734,2025-01-01 01:51:00+00:00,6.1,8.3,0.1,2024-12-31 20:51:00-05:00,20,12
3,72502014734,2025-01-01 02:20:00+00:00,5.6,7.8,0.1,2024-12-31 21:20:00-05:00,21,12
4,72502014734,2025-01-01 02:28:00+00:00,5.6,7.8,0.1,2024-12-31 21:28:00-05:00,21,12


In [ ]:
rows_per_station = merged.groupby("station_id").size().rename("n_rows").reset_index()
rows_per_station

missing_frac = merged.isna().mean().sort_values(ascending=False) #percentage of missing values per column
missing_frac.head(15).rename("missing_fraction").reset_index()

,index,missing_fraction
0,prcp_PRCP,0.230462
1,met_TMP,0.027420
2,met_DEW,0.027382
3,station_id,0.000000
4,date,0.000000
5,date_local,0.000000
6,hour_local,0.000000
7,month_local,0.000000
